# 02 — Clean & Standardize

Load the raw chart into DuckDB and clean it **in SQL** (workspace norm):
strip `$`/commas, cast to numeric, derive `decade` and the `inflation_multiple`
(adjusted / nominal). Result: the `films_adjusted` table.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from io import StringIO
from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, run_sql, register_source, save_interim

cfg = load_config('config.yaml')
con = get_connection(cfg)

# Reload the raw HTML parsed in 01 into a raw DuckDB table.
html = (Path(cfg['paths']['data_raw']) / 'bom_top_lifetime_adjusted_2022.html').read_text(encoding='utf-8')
raw = pd.read_html(StringIO(html))[0]
load_to_duckdb(raw, 'bom_raw', con)
print('bom_raw:', con.execute('SELECT COUNT(*) FROM bom_raw').fetchone()[0], 'rows')

## Clean in DuckDB
Money strings (`$1,895,421,694`) become `BIGINT`; `decade` uses integer
division; `inflation_multiple` = adjusted / nominal (how many times the film's
original take the adjusted figure represents).

In [ ]:
films = run_sql('''
    SELECT
        CAST("Rank" AS INTEGER)                                   AS rank_adjusted,
        "Title"                                                   AS title,
        CAST(REGEXP_REPLACE("Adj. Lifetime Gross", '[$,]', '', 'g') AS BIGINT) AS adjusted_gross,
        CAST(REGEXP_REPLACE("Lifetime Gross",       '[$,]', '', 'g') AS BIGINT) AS nominal_gross,
        CAST("Est. Num Tickets" AS BIGINT)                        AS est_tickets,
        CAST("Year" AS INTEGER)                                   AS release_year,
        (CAST("Year" AS INTEGER) // 10) * 10                      AS decade,
        ROUND(
            CAST(REGEXP_REPLACE("Adj. Lifetime Gross", '[$,]', '', 'g') AS DOUBLE)
            / NULLIF(CAST(REGEXP_REPLACE("Lifetime Gross", '[$,]', '', 'g') AS DOUBLE), 0),
            2) AS inflation_multiple
    FROM bom_raw
    ORDER BY adjusted_gross DESC
''', con)
load_to_duckdb(films, 'films_adjusted', con)
print(films.shape)
films.head(10)

## Quick sanity checks

In [ ]:
# The classics should dominate the adjusted board; recent blockbusters have
# the smallest inflation multiples (little time for ticket prices to rise).
print('Top adjusted:', films.iloc[0]['title'], films.iloc[0]['release_year'])
print('No nulls in key cols:',
      films[['adjusted_gross','nominal_gross','est_tickets','release_year']].notna().all().all())
films[['title','release_year','inflation_multiple']].sort_values('inflation_multiple').head(5)

## Clean the WORLDWIDE chart (domestic vs international split)
Same money-string cleaning; a bare `-` (no gross in a market) becomes NULL.
Produces `films_worldwide` with domestic/foreign/worldwide gross + the split.

In [ ]:
ww_html = (Path(cfg['paths']['data_raw']) / 'bom_ww_top_lifetime.html').read_text(encoding='utf-8')
ww_raw = pd.read_html(StringIO(ww_html))[0]
load_to_duckdb(ww_raw, 'bom_ww_raw', con)
ww = run_sql('''
    WITH cleaned AS (
        SELECT CAST("Rank" AS INTEGER) AS rank_worldwide, "Title" AS title,
               CAST("Year" AS INTEGER) AS release_year,
               TRY_CAST(NULLIF(REGEXP_REPLACE("Worldwide Lifetime Gross",'[$,]','','g'),'-') AS BIGINT) AS worldwide_gross,
               TRY_CAST(NULLIF(REGEXP_REPLACE("Domestic Lifetime Gross",'[$,]','','g'),'-') AS BIGINT) AS domestic_gross,
               TRY_CAST(NULLIF(REGEXP_REPLACE("Foreign Lifetime Gross",'[$,]','','g'),'-') AS BIGINT) AS foreign_gross
        FROM bom_ww_raw)
    SELECT rank_worldwide, title, worldwide_gross, domestic_gross, foreign_gross, release_year,
        ROUND(100.0*domestic_gross/NULLIF(worldwide_gross,0),1) AS domestic_pct,
        ROUND(100.0*foreign_gross /NULLIF(worldwide_gross,0),1) AS foreign_pct
    FROM cleaned ORDER BY worldwide_gross DESC''', con)
load_to_duckdb(ww, 'films_worldwide', con)
print('most international (lowest domestic %):')
print(ww.nsmallest(5,'domestic_pct')[['title','release_year','worldwide_gross','domestic_pct']].to_string(index=False))
ww.head()

## Load the genre data (ingested from TMDB in 01)

The TMDB genre lookup was done in `01-ingest` (that's where external sources are
pulled — see the API-key note there). Here we just **load** the ingested genre
records from `data/raw/tmdb_genres.parquet` into DuckDB — no API calls in the
cleaning stage. Builds `films_genre` (one row per film) and `film_genres_long`
(one row per film x genre, which drives the genre market-split analysis).

In [ ]:
genre_raw = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_genres.parquet')

films_genre = genre_raw.drop(columns=['genres']).rename(columns={'year':'release_year'})
load_to_duckdb(films_genre, 'films_genre', con)

long_rows = [{'title': r['title'], 'release_year': r['year'], 'genre': g}
             for r in genre_raw.to_dict('records') for g in r['genres']]
film_genres_long = pd.DataFrame(long_rows)
load_to_duckdb(film_genres_long, 'film_genres_long', con)
print(f'{films_genre["matched"].sum()}/{len(films_genre)} matched; {len(film_genres_long)} film-genre rows')
films_genre['primary_genre'].value_counts().head(8)

## Save interim + register provenance

In [ ]:
save_interim(films, cfg, 'films_adjusted.parquet')
save_interim(ww, cfg, 'films_worldwide.parquet')
save_interim(films_genre, cfg, 'films_genre.parquet')
save_interim(film_genres_long, cfg, 'film_genres_long.parquet')

register_source(con, 'films_adjusted',
    name='Box Office Mojo - Top Lifetime Adjusted Grosses (domestic)',
    url=cfg['sources']['bom_adjusted']['url'], license='Data (c) IMDb/Box Office Mojo.',
    notes='Domestic (US/Canada). Adjusted to 2022 $ via tickets x 2022 avg ticket price (ticket-price inflation, NOT CPI).',
    methodology='Tickets sold x reference-year avg price; lifetime totals include re-releases.',
    series_breaks='Nominal grosses are year-of-release dollars, not comparable across eras.')
register_source(con, 'films_worldwide',
    name='Box Office Mojo - Top Lifetime Grosses (Worldwide)',
    url=cfg['sources']['bom_worldwide']['url'], license='Data (c) IMDb/Box Office Mojo.',
    notes='Worldwide/domestic/foreign lifetime gross (NOMINAL $) + split. Domestic = US & Canada.',
    methodology='Studio-reported theatrical receipts; lifetime totals include re-releases.',
    series_breaks='Nominal dollars; worldwide totals favor recent wide-release films.')
register_source(con, 'films_genre',
    name='TMDB - film genres', url='https://www.themoviedb.org/',
    license='TMDB API; non-commercial attribution.',
    notes='Genre(s) per film matched by title+year. Multi-genre; used as a ratio in the market split.',
    methodology='TMDB /search/movie by title+year, first result.', series_breaks='')
print(con.execute('SELECT duckdb_table, source_name FROM _sources ORDER BY duckdb_table').df().to_string(index=False))

---
**Next:** `03-prepare.ipynb` packages the three exports + codebooks.

## Cleanup
Close the DuckDB connection so the write lock is released.

In [ ]:
con.close()
print('connection closed')